# Project 3 — Multimodal BEV Perception

**Notebook:** Deployment & Engineering  
**Models Evaluated:** Multimodal Fusion; Multimodal Hybrid Cross-Attention  
**Task:** Deployment Readiness & Inference Benchmarking  
**Dataset:** nuScenes-mini  
**Focus:** Latency, model size, GPU memory, TorchScript, and ONNX export

## Objective

This notebook evaluates the deployment readiness of the
two strongest multimodal BEV perception architectures
developed in this project:

- **Fusion Baseline**
- **Fusion + Hybrid X-Attention**

Unlike the previous notebooks, which focused on model
development and experimental evaluation, this notebook
examines practical engineering characteristics relevant
to real-world deployment.

The following analyses are performed:

- Parameter statistics
- Model size estimation
- GPU and CPU runtime benchmarking
- GPU memory profiling
- TorchScript export
- ONNX export
- Deployment comparison

The notebook concludes with an engineering summary
highlighting the trade-offs between segmentation
performance, computational efficiency, and deployment
readiness.

## Models Evaluated

### Fusion Baseline

The strongest-performing multimodal fusion model
developed during this project.

### Fusion + Hybrid X-Attention

An enhanced fusion architecture incorporating
cross-modal attention and hybrid feature fusion.

Both models are evaluated under identical deployment
conditions.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
%cd /content/drive/MyDrive/Colab Notebooks/project3_multimodal_bev_perception
!ls

Mounted at /content/drive
/content/drive/MyDrive/Colab Notebooks/project3_multimodal_bev_perception
checkpoints	data	    notebooks  README.md  scripts
colab_packages	dummy.onnx  nuscenes   reports	  src


In [ ]:
import os, sys
# --- System ---
sys.path.append('/content/drive/MyDrive/Colab Notebooks/project3_multimodal_bev_perception/colab_packages')


In [ ]:
import sys, os, importlib
# Make sure your src/ directory is visible to Python
project_root = "/content/drive/MyDrive/Colab Notebooks/project3_multimodal_bev_perception"
src_path = os.path.join(project_root, "src")

if src_path not in sys.path:
    sys.path.append(src_path)
    print(f" Added to sys.path: {src_path}")

# Clear any stale caches
importlib.invalidate_caches()

## Runtime Configuration

The deployment experiments are executed using PyTorch.

Unless otherwise noted, all runtime benchmarks use

- Batch size = 1
- Fixed sensor resolutions
- Warm-up iterations
- Repeated inference measurements

These settings reflect a realistic deployment scenario
for autonomous perception systems.

In [ ]:
# ============================================================
# Imports and Runtime Configuration
#
# This notebook focuses on deployment-oriented evaluation of
# trained multimodal perception models, including:
#
#   • Model statistics
#   • Runtime benchmarking
#   • Memory usage
#   • TorchScript export
#   • ONNX export
# ============================================================

import os
import time

import numpy as np
import torch

# ---------------------------------------------------
# Import trained models
# ---------------------------------------------------
from src.model.fusion_baseline import (
    FusionBaselineModel,
)

from src.model.fusion_hybrid_xattn import (
    FusionHybridXAttentionModel,
)

# ---------------------------------------------------
# Device configuration
# ---------------------------------------------------
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(f"Using device: {device}")

if torch.cuda.is_available():

    print(
        f"GPU: {torch.cuda.get_device_name(0)}"
    )


# ---------------------------------------------------
# Runtime configuration
#
# Enable cuDNN benchmarking to select the fastest
# convolution algorithms for fixed input sizes.
# ---------------------------------------------------
torch.backends.cudnn.benchmark = True

Using device: cuda
GPU: NVIDIA A100-SXM4-40GB


## Model Loading

Load the trained checkpoints used throughout the
deployment experiments.

Both models are restored in evaluation mode before
benchmarking.

In [ ]:
# ============================================================
# Load Trained Models
#
# Load the final trained checkpoints for the two models
# selected for deployment-oriented evaluation:
#
#   • Fusion Baseline
#   • Fusion + Hybrid X-Attention
# ============================================================

# ---------------------------------------------------
# Checkpoint locations
# ---------------------------------------------------
FUSION_CHECKPOINT = (
    "checkpoints/fusion_baseline_best.pth"
)

HYBRID_CHECKPOINT = (
    "checkpoints/fusion_hybrid_xattn_best.pth"
)


# ---------------------------------------------------
# Instantiate models
# ---------------------------------------------------
fusion_model = FusionBaselineModel().to(
    device,
)

hybrid_model = FusionHybridXAttentionModel().to(
    device,
)


# ---------------------------------------------------
# Load trained weights
# ---------------------------------------------------
fusion_model.load_state_dict(
    torch.load(
        FUSION_CHECKPOINT,
        map_location=device,
    )
)

hybrid_model.load_state_dict(
    torch.load(
        HYBRID_CHECKPOINT,
        map_location=device,
    )
)


# ---------------------------------------------------
# Evaluation mode
# ---------------------------------------------------
fusion_model.eval()

hybrid_model.eval()


# ---------------------------------------------------
# Model registry
#
# Used throughout the deployment notebook for
# benchmarking, export, and engineering evaluation.
# ---------------------------------------------------
models = {
    "Fusion Baseline": fusion_model,
    "Fusion + Hybrid X-Attention": hybrid_model,
}


print("Loaded deployment models")
print("------------------------")

for name in models:

    print(f"✓ {name}")

print("\nReady for deployment benchmarking.")

Loaded deployment models
------------------------
✓ Fusion Baseline
✓ Fusion + Hybrid X-Attention

Ready for deployment benchmarking.


## Parameter Statistics

Model complexity is summarized using

- Total parameters
- Trainable parameters
- Estimated model size

These metrics provide a first-order estimate of
deployment cost.

In [ ]:
# ============================================================
# Parameter Statistics
#
# Compare the computational complexity of the deployed models.
#
# Reported metrics
#   • Total parameters
#   • Trainable parameters
#   • Estimated model size (MB)
# ============================================================

# ---------------------------------------------------
# Utility function
# ---------------------------------------------------
def model_statistics(model):
    """
    Compute basic deployment statistics for a PyTorch model.

    Returns
    -------
    dict
        Dictionary containing:

        • Total parameters
        • Trainable parameters
        • Estimated model size (MB)
    """

    total_params = sum(
        p.numel()
        for p in model.parameters()
    )

    trainable_params = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    # ---------------------------------------------
    # Estimate model size assuming FP32 weights
    # (4 bytes per parameter)
    # ---------------------------------------------
    model_size_mb = (
        total_params * 4
    ) / (1024 ** 2)

    return {
        "Total Parameters": total_params,
        "Trainable Parameters": trainable_params,
        "Model Size (MB)": model_size_mb,
    }


# ---------------------------------------------------
# Compute statistics
# ---------------------------------------------------
deployment_summary = {}

print("=" * 72)
print("Deployment Statistics")
print("=" * 72)

for model_name, model in models.items():

    stats = model_statistics(model)

    deployment_summary[model_name] = stats

    print(f"\n{model_name}")
    print("-" * len(model_name))

    print(
        f"Total Parameters     : "
        f"{stats['Total Parameters']:,}"
    )

    print(
        f"Trainable Parameters : "
        f"{stats['Trainable Parameters']:,}"
    )

    print(
        f"Estimated Model Size : "
        f"{stats['Model Size (MB)']:.2f} MB"
    )

print("\nParameter analysis complete.")

Deployment Statistics

Fusion Baseline
---------------
Total Parameters     : 364,913
Trainable Parameters : 364,913
Estimated Model Size : 1.39 MB

Fusion + Hybrid X-Attention
---------------------------
Total Parameters     : 546,161
Trainable Parameters : 546,161
Estimated Model Size : 2.08 MB

Parameter analysis complete.


## Runtime Benchmark

Inference performance is evaluated on both GPU and CPU.

Measurements include

- Mean latency
- Latency variability
- Throughput (FPS)

Each benchmark consists of warm-up iterations followed
by repeated forward passes to obtain stable estimates.

In [ ]:
# ============================================================
# Runtime Benchmark
#
# Evaluate deployment-oriented inference performance of the
# trained multimodal perception models.
#
# Reported metrics
#   • Mean GPU latency (ms)
#   • Mean CPU latency (ms)
#   • Latency standard deviation (ms)
#   • Throughput (FPS)
#
# Timing is averaged over repeated forward passes following
# warm-up iterations to obtain stable runtime estimates.
# ============================================================

# ---------------------------------------------------
# Benchmark configuration
# ---------------------------------------------------
BATCH_SIZE = 1

WARMUP_ITERS = 20

BENCHMARK_ITERS = 100


# ---------------------------------------------------
# Representative deployment inputs
# ---------------------------------------------------
rgb_example = torch.randn(
    BATCH_SIZE,
    3,
    256,
    256,
)

bev_example = torch.randn(
    BATCH_SIZE,
    1,
    200,
    200,
)

imu_example = torch.randn(
    BATCH_SIZE,
    10,
    6,
)


# ---------------------------------------------------
# Benchmark function
# ---------------------------------------------------
def benchmark_model(
    model,
    device,
):
    """
    Benchmark inference latency and throughput.

    Parameters
    ----------
    model : nn.Module
        Trained model.

    device : torch.device
        Target execution device.

    Returns
    -------
    tuple

        (
            mean_latency_ms,
            std_latency_ms,
            fps,
        )
    """

    model = model.to(device)

    model.eval()

    rgb = rgb_example.to(device)

    bev = bev_example.to(device)

    imu = imu_example.to(device)

    # -------------------------------------------------
    # Warm-up
    # -------------------------------------------------
    with torch.inference_mode():

        for _ in range(WARMUP_ITERS):

            _ = model(
                rgb,
                bev,
                imu,
            )

    # -------------------------------------------------
    # Timed benchmark
    # -------------------------------------------------
    timings = []

    with torch.inference_mode():

        for _ in range(BENCHMARK_ITERS):

            if device.type == "cuda":

                torch.cuda.synchronize()

            start = time.perf_counter()

            _ = model(
                rgb,
                bev,
                imu,
            )

            if device.type == "cuda":

                torch.cuda.synchronize()

            end = time.perf_counter()

            timings.append(
                end - start
            )

    timings = np.asarray(
        timings
    )

    mean_latency = timings.mean()

    std_latency = timings.std()

    fps = 1.0 / mean_latency

    return (

        mean_latency * 1000,

        std_latency * 1000,

        fps,
    )


# ============================================================
# GPU Benchmark
# ============================================================
if torch.cuda.is_available():

    print("=" * 72)
    print("GPU Runtime Benchmark")
    print("=" * 72)

    print(f"Batch size         : {BATCH_SIZE}")
    print(f"Warm-up iterations : {WARMUP_ITERS}")
    print(f"Benchmark runs     : {BENCHMARK_ITERS}")

    gpu_device = torch.device("cuda")

    for model_name, model in models.items():

        latency_ms, latency_std, fps = benchmark_model(
            model,
            gpu_device,
        )

        deployment_summary[model_name][
            "GPU Latency (ms)"
        ] = latency_ms

        deployment_summary[model_name][
            "GPU Latency Std (ms)"
        ] = latency_std

        deployment_summary[model_name][
            "GPU FPS"
        ] = fps

        print(f"\n{model_name}")
        print("-" * len(model_name))

        print(
            f"Mean Latency : {latency_ms:.2f} ms"
        )

        print(
            f"Latency Std  : {latency_std:.3f} ms"
        )

        print(
            f"Throughput   : {fps:.2f} FPS"
        )

else:

    print(
        "CUDA not available — GPU benchmark skipped."
    )


# ============================================================
# CPU Benchmark
# ============================================================
print("\n" + "=" * 72)
print("CPU Runtime Benchmark")
print("=" * 72)

print(f"Batch size         : {BATCH_SIZE}")
print(f"Warm-up iterations : {WARMUP_ITERS}")
print(f"Benchmark runs     : {BENCHMARK_ITERS}")

cpu_device = torch.device("cpu")

for model_name, model in models.items():

    latency_ms, latency_std, fps = benchmark_model(
        model,
        cpu_device,
    )

    deployment_summary[model_name][
        "CPU Latency (ms)"
    ] = latency_ms

    deployment_summary[model_name][
        "CPU Latency Std (ms)"
    ] = latency_std

    deployment_summary[model_name][
        "CPU FPS"
    ] = fps

    print(f"\n{model_name}")
    print("-" * len(model_name))

    print(
        f"Mean Latency : {latency_ms:.2f} ms"
    )

    print(
        f"Latency Std  : {latency_std:.3f} ms"
    )

    print(
        f"Throughput   : {fps:.2f} FPS"
    )

print("\nRuntime benchmarking complete.")

GPU Runtime Benchmark
Batch size         : 1
Warm-up iterations : 20
Benchmark runs     : 100

Fusion Baseline
---------------
Mean Latency : 2.56 ms
Latency Std  : 0.154 ms
Throughput   : 389.89 FPS

Fusion + Hybrid X-Attention
---------------------------
Mean Latency : 3.87 ms
Latency Std  : 0.104 ms
Throughput   : 258.18 FPS

CPU Runtime Benchmark
Batch size         : 1
Warm-up iterations : 20
Benchmark runs     : 100

Fusion Baseline
---------------
Mean Latency : 64.69 ms
Latency Std  : 7.689 ms
Throughput   : 15.46 FPS

Fusion + Hybrid X-Attention
---------------------------
Mean Latency : 68.84 ms
Latency Std  : 6.101 ms
Throughput   : 14.53 FPS

Runtime benchmarking complete.


## GPU Memory Profiling

Peak GPU memory allocation during inference is measured
to estimate deployment resource requirements.

In [ ]:
# ============================================================
# GPU Memory Usage
#
# Measure peak GPU memory consumption during inference.
#
# Reported metric
#   • Peak GPU memory (MB)
#
# Peak memory provides insight into deployment resource
# requirements and complements latency benchmarking.
# ============================================================

if torch.cuda.is_available():

    print("=" * 72)
    print("GPU Memory Usage")
    print("=" * 72)

    gpu_device = torch.device("cuda")

    # Representative deployment inputs
    rgb_gpu = rgb.to(gpu_device)
    bev_gpu = bev.to(gpu_device)
    imu_gpu = imu.to(gpu_device)

    for model_name, model in models.items():

        model = model.to(gpu_device)
        model.eval()

        # ---------------------------------------------
        # Reset memory statistics
        # ---------------------------------------------
        torch.cuda.empty_cache()

        torch.cuda.reset_peak_memory_stats(
            gpu_device
        )

        with torch.no_grad():

            _ = model(
                rgb_gpu,
                bev_gpu,
                imu_gpu,
            )

        torch.cuda.synchronize()

        peak_memory_mb = (
            torch.cuda.max_memory_allocated(
                gpu_device
            )
            / (1024 ** 2)
        )

        deployment_summary[model_name][
            "Peak GPU Memory (MB)"
        ] = peak_memory_mb

        print(f"\n{model_name}")
        print("-" * len(model_name))

        print(
            f"Peak GPU Memory : "
            f"{peak_memory_mb:.2f} MB"
        )

else:

    print(
        "CUDA not available — "
        "GPU memory profiling skipped."
    )

GPU Memory Usage

Fusion Baseline
---------------
Peak GPU Memory : 90.33 MB

Fusion + Hybrid X-Attention
---------------------------
Peak GPU Memory : 95.43 MB


## TorchScript Export

The trained models are exported to TorchScript format
and reloaded to verify successful deployment within the
PyTorch ecosystem.

In [ ]:
# ============================================================
# TorchScript Export
#
# Export the trained models to TorchScript format and
# verify that the exported models can be loaded and
# executed successfully.
#
# Generated artifacts
#   • deployment/fusion_baseline.ts
#   • deployment/fusion_hybrid_xattn.ts
# ============================================================

import warnings

# ---------------------------------------------------
# Output directory
# ---------------------------------------------------
EXPORT_DIR = "deployment"

os.makedirs(
    EXPORT_DIR,
    exist_ok=True,
)

# ---------------------------------------------------
# Example deployment inputs
# ---------------------------------------------------
rgb_example = rgb.to(device)
bev_example = bev.to(device)
imu_example = imu.to(device)

# ---------------------------------------------------
# Export configuration
# ---------------------------------------------------
export_configs = [

    (
        "Fusion Baseline",
        fusion_model,
        "fusion_baseline.ts",
    ),

    (
        "Fusion + Hybrid X-Attention",
        hybrid_model,
        "fusion_hybrid_xattn.ts",
    ),
]

print("=" * 72)
print("TorchScript Export")
print("=" * 72)

for model_name, model, filename in export_configs:

    model.eval()

    export_path = os.path.join(
        EXPORT_DIR,
        filename,
    )

    # ---------------------------------------------
    # Export TorchScript model
    #
    # Suppress expected TracerWarnings caused by
    # tensor-dependent shape checks during tracing.
    # These warnings are expected because deployment
    # uses fixed sensor resolutions.
    # ---------------------------------------------
    with warnings.catch_warnings():

        warnings.simplefilter(
            "ignore",
            category=torch.jit.TracerWarning,
        )

        traced_model = torch.jit.trace(
            model,
            (
                rgb_example,
                bev_example,
                imu_example,
            ),
        )

    # ---------------------------------------------
    # Reload exported model
    # ---------------------------------------------
    loaded_model = torch.jit.load(
        export_path,
        map_location=device,
    )

    loaded_model.eval()

    # ---------------------------------------------
    # Verify inference
    # ---------------------------------------------
    with torch.no_grad():

        output = loaded_model(
            rgb_example,
            bev_example,
            imu_example,
        )

    deployment_summary[model_name][
        "TorchScript"
    ] = True

    deployment_summary[model_name][
        "TorchScript File"
    ] = export_path

    print(f"\n✓ {model_name}")

    print(
        f"Saved to : {export_path}"
    )

    print(
        f"Verified output shape : "
        f"{tuple(output.shape)}"
    )

print("\nTorchScript export completed successfully.")

TorchScript Export

✓ Fusion Baseline
Saved to : deployment/fusion_baseline.ts
Verified output shape : (1, 1, 512, 512)

✓ Fusion + Hybrid X-Attention
Saved to : deployment/fusion_hybrid_xattn.ts
Verified output shape : (1, 1, 512, 512)

TorchScript export completed successfully.


## ONNX Export

The trained models are exported to ONNX to demonstrate
framework-independent deployment compatibility.

In [ ]:
# !pip install -q onnx onnxscript

In [ ]:
# ============================================================
# ONNX Export
#
# Export the trained models to ONNX format for
# framework-independent deployment.
#
# Generated artifacts
#   • deployment/fusion_baseline.onnx
#   • deployment/fusion_hybrid_xattn.onnx
#
# If ONNX export fails for a model, the notebook
# continues execution and reports the reason.
# ============================================================

# ---------------------------------------------------
# Output directory
# ---------------------------------------------------
EXPORT_DIR = "deployment"

os.makedirs(
    EXPORT_DIR,
    exist_ok=True,
)

# ---------------------------------------------------
# Example deployment inputs
# ---------------------------------------------------
rgb_example = rgb.to(device)
bev_example = bev.to(device)
imu_example = imu.to(device)

# ---------------------------------------------------
# Export configuration
# ---------------------------------------------------
export_configs = [

    (
        "Fusion Baseline",
        fusion_model,
        "fusion_baseline.onnx",
    ),

    (
        "Fusion + Hybrid X-Attention",
        hybrid_model,
        "fusion_hybrid_xattn.onnx",
    ),
]

print("=" * 72)
print("ONNX Export")
print("=" * 72)

for model_name, model, filename in export_configs:

    model.eval()

    export_path = os.path.join(
        EXPORT_DIR,
        filename,
    )

    try:

        torch.onnx.export(
            model,
            (
                rgb_example,
                bev_example,
                imu_example,
            ),
            export_path,

            export_params=True,

            opset_version=18,

            do_constant_folding=True,

            input_names=[
                "rgb",
                "bev",
                "imu",
            ],

            output_names=[
                "occupancy_logits",
            ]

        )

        deployment_summary[model_name][
            "ONNX"
        ] = True

        deployment_summary[model_name][
            "ONNX File"
        ] = export_path

        print(f"\n✓ {model_name}")

        print(
            f"Saved to : {export_path}"
        )

    except Exception as e:

        deployment_summary[model_name][
            "ONNX"
        ] = False

        deployment_summary[model_name][
            "ONNX File"
        ] = "Export Failed"

        print(f"\n✗ {model_name}")

        print(
            "ONNX export failed."
        )

        print(
            f"Reason: {e}"
        )

print("\nONNX export completed.")

ONNX Export
[torch.onnx] Obtain model graph for `FusionBaselineModel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `FusionBaselineModel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅

✓ Fusion Baseline
Saved to : deployment/fusion_baseline.onnx
[torch.onnx] Obtain model graph for `FusionHybridXAttentionModel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `FusionHybridXAttentionModel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...


[torch.onnx] Optimize the ONNX graph... ✅

✓ Fusion + Hybrid X-Attention
Saved to : deployment/fusion_hybrid_xattn.onnx

ONNX export completed.


## Deployment Summary

The table below summarizes the principal engineering
characteristics of the final perception models.

It combines model complexity, segmentation performance,
runtime efficiency, memory usage, and deployment
compatibility into a single comparison.

In [ ]:
# ============================================================
# Deployment Summary
#
# Summarize the deployment characteristics of the final
# perception models, including model complexity,
# segmentation performance, runtime efficiency, memory
# consumption, and exportability.
# ============================================================

import pandas as pd

# ---------------------------------------------------
# Final segmentation performance
#
# Mean IoU values obtained during quantitative
# evaluation of each trained model.
# ---------------------------------------------------
FINAL_IOU = {
    "Fusion Baseline": 0.7150,
    "Fusion + Hybrid X-Attention": 0.6017,
}

# ---------------------------------------------------
# Build deployment summary table
# ---------------------------------------------------
summary_rows = []

for model_name, stats in deployment_summary.items():

    summary_rows.append({

        "Model":
            model_name,

        "Parameters":
            f"{stats['Total Parameters']:,}",

        "Model Size (MB)":
            f"{stats['Model Size (MB)']:.2f}",

        "Mean IoU":
            f"{FINAL_IOU.get(model_name, np.nan):.4f}",

        "GPU Latency (ms)":
            f"{stats.get('GPU Latency (ms)', np.nan):.2f}",

        "GPU FPS":
            f"{stats.get('GPU FPS', np.nan):.2f}",

        "CPU Latency (ms)":
            f"{stats.get('CPU Latency (ms)', np.nan):.2f}",

        "CPU FPS":
            f"{stats.get('CPU FPS', np.nan):.2f}",

        "Peak GPU Memory (MB)":
            f"{stats.get('Peak GPU Memory (MB)', np.nan):.2f}",

        "TorchScript":
            "✓"
            if stats.get("TorchScript", False)
            else "✗",

        "ONNX":
            "✓"
            if stats.get("ONNX", False)
            else "✗",
    })

deployment_df = pd.DataFrame(
    summary_rows
)

print("=" * 120)
print("Deployment Summary")
print("=" * 120)

display(
    deployment_df.style.hide(axis="index")
)

Deployment Summary


Model,Parameters,Model Size (MB),Mean IoU,GPU Latency (ms),GPU FPS,CPU Latency (ms),CPU FPS,Peak GPU Memory (MB),TorchScript,ONNX
Fusion Baseline,"364,913",1.39,0.7150,2.56,389.89,64.69,15.46,90.33,✓,✓
Fusion + Hybrid X-Attention,"546,161",2.08,0.6017,3.87,258.18,68.84,14.53,95.43,✓,✓


# Engineering Conclusions

The deployment analysis demonstrates that both models
are suitable for real-time multimodal BEV perception.

### Fusion Baseline

- Highest segmentation accuracy
- Lowest GPU latency
- Highest throughput
- Smallest memory footprint

### Fusion + Hybrid X-Attention

- Explores a more expressive cross-modal fusion strategy
- Remains lightweight and deployable
- Incurs only modest increases in runtime and memory

Overall, the experiments illustrate the engineering
trade-offs between architectural complexity and
deployment efficiency. While the Hybrid X-Attention
model introduces additional representational capacity,
the Fusion Baseline remains the preferred model for
this dataset due to its superior accuracy and runtime
performance.